In [0]:
%run /Workspace/Users/marcoaurelioreislima@gmail.com/databricks-playground/projects/data_generator/streaming/0_data_generator

In [0]:
from datetime import datetime as dt
from random import randint
from uuid import uuid4
import pytest
import faker
from pandas import DataFrame as PandasDF
import numpy as np
import time
import pandas as pd

from pyspark.sql.functions import lit
from rand_engine.main.data_generator import DataGenerator
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore
from rand_engine.core.distinct_utils import DistinctUtils

class CDCGenerator:

  def __init__(self, footprint: FakeCustomer):
    self.footprint = footprint
    self.min_secs, self.max_secs = 10, 20
    self.min_size, self.max_size = 100, 200

  def config_file_props(self, base_path: str, file_name, ext):
    self.base_path = f"{base_path}/{file_name}/{ext}"
    self.file_name = file_name
    self.ext = ext
    return self

  def config_period(self, min_secs, max_secs):
    self.min_secs = min_secs
    self.max_secs = max_secs
    return self

  def config_size(self, min_size, max_size):
    self.min_size = min_size
    self.max_size = max_size
    return self

  def generate_sample(self, size: int=100):
    return (
      DataGenerator(self.footprint.metadata())
        .generate_pandas_df(size, transformer=self.footprint.transformer())
        .get_df()
    )

  def list_files(self):
    assert self.base_path, "Base path not configured. Run the method config_file_props."
    dbutils.fs.mkdirs(self.base_path)
    display(dbutils.fs.ls(self.base_path))

  def delete_files(self):
    assert self.base_path, "Base path not configured. Run the method config_file_props."
    dbutils.fs.rm(f"dbfs:{self.base_path}", True)

  def generate_inserts(self):
    file_path = f"{self.base_path}/{self.file_name}_{str(uuid4())[:8]}.{self.ext}"
    rand_size = randint(self.min_size, self.max_size)
    _ = (
      DataGenerator(self.footprint.metadata()) \
        .generate_pandas_df(rand_size, transformer=self.footprint.transformer(operation="INSERT"))
        .write() \
        .mode("overwrite") \
        .format(f"{self.ext}") \
        .option("compression", None) \
        .load(file_path)
    )
    print(f"File {file_path} created with {rand_size} records.")


  def generate_updates(self, base_path, sample=0.02):
    file_compl = str(uuid4())[:8]
    file_path = f"{base_path}/customers_{file_compl}.{self.ext}"
    df = spark.read.format("json").load(base_path)
    colunas = [col for col in df.columns if col not in ("operation", "user_id")]
    df_ids_inserted = df.select("user_id").filter("operation = 'INSERT'").distinct()


  def generate_deletes(self, pk_cols=["user_id"], sample=0.02):
    file_path = f"{self.base_path}/{self.file_name}_{str(uuid4())[:8]}.{self.ext}"
    df = spark.read.format("json").load(self.base_path)
    colunas = [col for col in df.columns if col not in ["operation", *pk_cols]]
    df_ids_inserted = df.select(*pk_cols).filter("operation = 'INSERT'").distinct()
    df_ids_deleted = df_ids_inserted.select(*pk_cols).filter("operation = 'DELETE'").distinct()
    df_ids_to_delete = df_ids_inserted.join(df_ids_deleted, on=pk_cols, how="leftanti") 
    df_ids_to_delete = df_ids_to_delete.sample(sample)
    for coluna in colunas:
      df_ids_to_delete = df_ids_to_delete.withColumn(coluna, lit(None))
    df_ids_to_delete = df_ids_to_delete.withColumn("operation", lit("DELETE"))
    pandas_df = df_ids_to_delete.toPandas()
    pandas_df.to_json(file_path, orient="records", lines=True)


In [0]:
BASE_PATH, FILE_NAME, EXT = ("/Volumes/prd/demo_volumes/rand_engine_data/cdc", "customers", "json")
#dbutils.fs.rm(f"dbfs:{base_path}", True)
cdc_generator = CDCGenerator(FakeCustomer()).config_file_props(BASE_PATH, file_name=FILE_NAME, ext=EXT)
cdc_generator.generate_inserts()
cdc_generator.generate_deletes()
cdc_generator.list_files()

In [0]:
# %sql

# APPLY CHANGES INTO STREAM target
# FROM STREAM source
# KEYS (user_id)
# APPLY AS DELETE WHEN operation = "DELETE"
# SEQUENCE timestamp_datetime
# COLUMNS * EXCEPT (timestamp, _rescued_data, operation)
# STORED AS SCD TYPE 2